# Colab T4 Remote Setup for VS Code

This notebook sets up a Colab T4 GPU runtime that VS Code can connect to via SSH tunnel.

**How it works:**
1. Colab starts an SSH server + cloudflared tunnel
2. VS Code connects via Remote-SSH extension
3. You edit & run code on Colab's T4 GPU directly from VS Code
4. Use git to sync changes back to your local machine

**Prerequisites (local machine):**
- VS Code with **Remote - SSH** extension installed
- `cloudflared` installed locally: `brew install cloudflare/cloudflare/cloudflared`

**Steps:**
1. Open this notebook in Google Colab (Runtime → Change runtime type → T4 GPU)
2. Run all cells
3. Copy the SSH config printed at the end
4. In VS Code: Cmd+Shift+P → "Remote-SSH: Connect to Host" → select `colab`

In [ ]:
# Cell 1: Verify GPU
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# Cell 2: Set up SSH server + cloudflared tunnel
import subprocess
import os
import secrets
import string

# Generate a random password
password = ''.join(secrets.choice(string.ascii_letters + string.digits) for _ in range(20))

# Install and configure SSH
subprocess.run(['apt-get', 'install', '-qq', '-o=Dpkg::Use-Pty=0', 'openssh-server'], 
               capture_output=True)

# Set root password
subprocess.run(['bash', '-c', f'echo "root:{password}" | chpasswd'], capture_output=True)

# Configure SSH
with open('/etc/ssh/sshd_config', 'a') as f:
    f.write('\nPermitRootLogin yes\n')
    f.write('PasswordAuthentication yes\n')

# Start SSH
subprocess.run(['service', 'ssh', 'start'], capture_output=True)

# Install cloudflared
subprocess.run(['bash', '-c', 
    'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb '
    '&& dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1'], 
    capture_output=True)

print(f"SSH password: {password}")
print("SSH server started. Starting cloudflared tunnel...")

In [ ]:
# Cell 3: Start cloudflared tunnel (this cell keeps running)
import subprocess
import re
import time
import threading

# Start cloudflared in background
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'ssh://localhost:22', '--logfile', '/tmp/cloudflared.log'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Wait for tunnel URL
time.sleep(8)

with open('/tmp/cloudflared.log', 'r') as f:
    log = f.read()

# Extract the tunnel hostname
match = re.search(r'https://([a-zA-Z0-9-]+\.trycloudflare\.com)', log)
if match:
    hostname = match.group(1)
    print("=" * 60)
    print("TUNNEL ACTIVE")
    print("=" * 60)
    print(f"\nHostname: {hostname}")
    print(f"Password: {password}")
    print()
    print("Add this to your LOCAL ~/.ssh/config:")
    print("-" * 40)
    print(f"""Host colab""")
    print(f"""  HostName {hostname}""")
    print(f"""  User root""")
    print(f"""  ProxyCommand cloudflared access ssh --hostname %h""")
    print(f"""  StrictHostKeyChecking no""")
    print("-" * 40)
    print()
    print("Then in VS Code: Cmd+Shift+P → 'Remote-SSH: Connect to Host' → colab")
    print(f"Enter password when prompted: {password}")
    print()
    print("⚠️  KEEP THIS CELL RUNNING — closing it kills the tunnel.")
else:
    print("ERROR: Could not find tunnel URL. Check log:")
    print(log)

In [ ]:
# Cell 4: Clone repo + install dependencies (run AFTER connecting via SSH, or here)
# Mount Google Drive for large model files
from google.colab import drive
drive.mount('/content/drive')

# Clone the project repo
!cd /root && git clone https://github.com/anvay-cpu/voice-analysis-pipeline.git project 2>/dev/null || \
    (cd /root/project && git pull)

print("\nRepo cloned to /root/project")

In [ ]:
# Cell 5: Install all project dependencies
!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets librosa parselmouth speechbrain
!pip install -q opencv-python mediapipe ultralytics
!pip install -q scikit-learn scipy pyyaml tqdm
!pip install -q yt-dlp moviepy

# Verify CUDA
import torch
print(f"\nPyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Symlink Drive models
import os
drive_models = '/content/drive/MyDrive/voice_pipeline_models'
local_models = '/root/project/models'
os.makedirs(local_models, exist_ok=True)

if os.path.exists(drive_models):
    for item in os.listdir(drive_models):
        src = os.path.join(drive_models, item)
        dst = os.path.join(local_models, item)
        if not os.path.exists(dst):
            os.symlink(src, dst)
            print(f"Linked: {item}")

print("\n✓ Environment ready. Open /root/project in VS Code Remote-SSH.")

In [ ]:
# Cell 6: Keep-alive (prevents Colab from disconnecting)
# Run this cell to keep the session active
import time
import IPython.display

print("Keep-alive running. Session will stay active.")
print("Interrupt this cell (stop button) when you're done.")

i = 0
while True:
    time.sleep(60)
    i += 1
    IPython.display.clear_output(wait=True)
    print(f"Keep-alive running... ({i} min)")
    print(f"Tunnel hostname: {hostname}")
    print(f"Password: {password}")